# Feature Analysis — Dimensionality &amp; Structure

> **SPI Time Series** · Notebook `02` · Understand what the model sees
> **⚠️ Dev configs** — Uses `*_dev.yaml` configs for fast extraction

This notebook uses the **same public API as the CLI** to extract features from
different pipeline configurations and analyse their structure with PCA.

**What you'll find here:**
- Build a feature extractor from a dev config (same as `main.py`)
- Run the pipeline's preprocessor → splitter → extractor stages
- Compare dimensionality: log-based vs. one-hot encoded features
- PCA — explained variance, cumulative elbow plot, target projection

## 1. Build a Pipeline from Config

Same approach as the CLI: load a YAML config, build the pipeline with
`PipelineBuilder.from_config()`, and wire in a feature extractor.
We use dev configs for fast iteration — swap to the production configs
for a full run.


In [ ]:
from pathlib import Path

from spi_time_series import PipelineBuilder
from spi_time_series.config.schema import RunConfig
from spi_time_series.data.schemas import RawData
from spi_time_series.features.extraction import extract_features_builder
from spi_time_series.features.log_based_features import (
    BasicControlFlowFeatures,
    InteractionFeatures,
    OfferFeatures,
    WaitingStateFeatures,
)
from spi_time_series.features.targets import outcome_target

# Load config and build feature extractor (same logic as CLI's main.py)
config = RunConfig.from_yaml(Path("../configs/classification_dev.yaml"))
feature_list = []
for name in config.features.enabled_features:
    if name == "BasicControlFlowFeatures":
        feature_list.append(
            BasicControlFlowFeatures(one_hot_encode_categorical=False)
        )
    elif name == "OfferFeatures":
        feature_list.append(OfferFeatures())
    elif name == "InteractionFeatures":
        feature_list.append(InteractionFeatures())
    elif name == "WaitingStateFeatures":
        feature_list.append(WaitingStateFeatures())

extractor = extract_features_builder(feature_list, outcome_target)

# Build pipeline — same builder as CLI
pipeline = (
    PipelineBuilder.from_config(config)
    .with_feature_extractor(extractor)
    .build()
)

print(
    f"Pipeline ready: task='{pipeline.task}', "
    f"{len(feature_list)} feature group(s) enabled"
)

## 2. Extract Features

Run the pipeline's public stages — preprocessor, splitter, feature extractor —
just like `Pipeline.fit()` does internally. No custom wrappers.


In [ ]:
# Run pipeline stages — these are public attributes on Pipeline
raw = RawData(event_log=pipeline.dataset.log)
cleaned = pipeline.preprocessor(raw)
preprocessed = pipeline.splitter(cleaned)
features = pipeline.feature_extractor(preprocessed)

X = features.X_train.select_dtypes(include="number")

print(f"Feature matrix: {X.shape[0]:,} rows × {X.shape[1]:,} columns")
print(f"\nFeature groups:")
for name in config.features.enabled_features:
    count = sum(
        1
        for c in X.columns
        if f"__{name}" in c or (c.startswith(name) and "__" in c)
    )
    if count:
        print(f"  {name}: {count} column(s)")
print(
    f"  other: {X.shape[1] - sum(1 for c in X.columns for g in config.features.enabled_features if g in c)}"
)

## 3. PCA — Explained Variance

Fit PCA on the feature matrix to see how much variance the first few
principal components capture. This helps tune `pca_config.keep_variability`
in your YAML config.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

%matplotlib inline

# Standardize and fit PCA
X_scaled = StandardScaler().fit_transform(X)
pca = PCA().fit(X_scaled)

# Cumulative variance & component count to reach thresholds
cumsum = np.cumsum(pca.explained_variance_ratio_)
n_80 = int(np.searchsorted(cumsum, 0.80) + 1)
n_95 = int(np.searchsorted(cumsum, 0.95) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(range(1, 21), pca.explained_variance_ratio_[:20])
ax1.set_title("Explained Variance per Component (top 20)")
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Variance Ratio")

ax2.plot(range(1, len(cumsum) + 1), cumsum, marker=".", markersize=2)
ax2.axhline(0.95, color="#e74c3c", linestyle="--", label=f"95 % ({n_95} PCs)")
ax2.axhline(0.80, color="#f39c12", linestyle="--", label=f"80 % ({n_80} PCs)")
ax2.legend()
ax2.set_title("Cumulative Explained Variance")
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Variance Ratio")
ax2.set_ylim(0, 1.02)

fig.tight_layout()
plt.show()

print(f"Components needed: {n_80} for 80 %, {n_95} for 95 % of variance")
print(
    f"Recommended pca_config.keep_variability: 0.80"
    if n_80 < X.shape[1] * 0.3
    else "Consider higher keep_variability or no PCA"
)